In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-02 06:43:04.328418: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-02 06:43:05.012171: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [3]:
config = {
    "lib": "tensorflow",
    "mode": 'local',
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [7]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [8]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

Rain is initialized
Provisioner: Creating coordinator
Coordinator initialized successfully
LocalProvisioner is initialized


In [9]:
model = rain.train_centralized_sync()

2023-07-02 06:43:06,818 [INFO] [LogService] provisioner is serving
2023-07-02 06:43:06,822 [DEBUG] [LogService] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers


Rain: Creating workers
Provisioner: Starting coordinator
coordinator is serving
Coordinator: sending the num of workers to the provisioner
divider received: Success receiving the number of workers from provisioner
LocalProvisioner: Creating workers
Worker is running on port: 50151
Worker is running on port: 50152
Worker is running on port: 50153
[Created workers]
 IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]
Transceiver is serving
Rain: Sending data to workers
divider is sending data to the coordinator
 divider received: File received successfully from coordinator
 divider received: File received successfully from coordinator
 divider received: File received successfully from coordinator
 divider received: File received successfully from coordinator


2023-07-02 06:43:19,377 [DEBUG] [LogService] Received '' from the coordinator to send status
2023-07-02 06:43:19,377 [DEBUG] [LogService] Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]


 divider received: File received successfully from coordinator
 divider received: File received successfully from coordinator
Rain: Training
Starting iteration 1/3
sending file:  ../../../Divider/divider/data/1.pkl
divider is sending information file to the coordinator
divider received: File received successfully from coordinator
divider begins the iteration
start loop
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
executing command:  python3 ../../../Worker/Algo.py 

2023-07-02 06:43:36.681035: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 06:43:36.688433: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 06:43:36.692407: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


...

Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 8ms/step - loss: 0.2541 - accuracy: 0.9229
Epoch 2/2
157/157 [==============================] - 2s 8ms/step - loss: 0.2611 - accuracy: 0.9209
Epoch 2/2
157/157 [==============================] - 1s 8ms/step - loss: 0.2080 - accuracy: 0.9379
sending data to coordinator
157/157 [==============================] - 1s 8ms/step - loss: 0.2067 - accuracy: 0.9387
sending data to coordinator
coordinator received: Executed! from worker 
coordinator received: Executed! from worker 
sending data to coordinator
coordinator received: Executed! from worker 
thread 1 is done
Downloaded ../../../Coordinator/coord/data/1_2_trained.pkl in coordinator
thread 2 is done
Downloaded ../../../Coordinator/coord/data/2_2_trained.pkl in coordinator
thread 3 is done
Downloaded ../../../Coordinator/coord/data/3_2_trained.pkl in coordinator
coordinator received: Success! from divider 
coordinator received: Success! from divider 


2023-07-02 06:44:09,280 [DEBUG] [LogService] Received '' from the coordinator to send status
2023-07-02 06:44:09,281 [DEBUG] [LogService] Workers
IPs : ['127.0.0.1', '127.0.0.1', '127.0.0.1'], ports: [50151, 50152, 50153], statuses: [1, 1, 1], IDs : [1, 2, 3]


coordinator received: Success! from divider 
divider received: one loop is done from coordinator
Iteration 2/3 complete.
Starting iteration 3/3
sending file:  ../../../Divider/divider/data/3.pkl
divider is sending information file to the coordinator
divider received: File received successfully from coordinator
divider begins the iteration
start loop
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
coordinator received: File downloaded successfully from worker 
executing command:  python3 ../../../Worker/Algo.py 1 ../../../Worker/wo

2023-07-02 06:44:24.264993: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 06:44:24.273675: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2023-07-02 06:44:24.275258: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Epoch 1/2
Epoch 1/2
Epoch 1/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1919 - accuracy: 0.9445
Epoch 2/2
157/157 [==============================] - 2s 8ms/step - loss: 0.1818 - accuracy: 0.9449
Epoch 2/2
157/157 [==============================] - 1s 9ms/step - loss: 0.1582 - accuracy: 0.9530
sending data to coordinator
157/157 [==============================] - 1s 8ms/step - loss: 0.1533 - accuracy: 0.9545
sending data to coordinator
157/157 [==============================] - 1s 8ms/step - loss: 0.1443 - accuracy: 0.9557
sending data to coordinator
coordinator received: Executed! from worker 
coordinator received: Executed! from worker 
thread 1 is done
coordinator received: Executed! from worker 
Downloaded ../../../Coordinator/coord/data/1_3_trained.pkl in coordinator
thread 2 is done
Downloaded ../../../Coordinator/coord/data/2_3_trained.pkl in coordinator
thread 3 is done
Downloaded ../../../Coordinator/coord/data/3_3_trained.pkl in coordinator
coordinator r

In [10]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0999 - accuracy: 0.9684

Test accuracy: 96.8%
